# 02 · Cleaning and enrichment

### Session 1 support notebook | Safiri Telecom churn case

---

Cleaning is where leakage is most often introduced, and where it is hardest to see,
because every individual step looks harmless.

The rule that prevents almost all of it:

> **Any number you learn from the data must be learned from the training data
> alone, then applied unchanged to the test data.**

A median used to fill gaps. A quantile used to cap outliers. A threshold that
defines "heavy user". Each is a number learned from data, and each one leaks if you
learn it from everything.

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

Everyone completes READ and FILL. If a cell errors, skip it and keep moving.

## 0. Setup, data, and some realistic dirt

In [ ]:
import numpy as np
import pandas as pd

def make_panel(seed=42, n_subscribers=9000, n_months=8):
    """Simulate a monthly prepaid subscriber panel for Safiri Telecom."""
    rng = np.random.default_rng(seed)
    n = n_subscribers
    region  = rng.choice(["Nairobi","Coast","Western","Rift Valley","Eastern"], n, p=[.34,.16,.18,.20,.12])
    plan    = rng.choice(["Prepaid Basic","Prepaid Plus","Data Bundle"], n, p=[.55,.28,.17])
    handset = rng.choice(["Feature","Smart-Entry","Smart-Mid","Smart-High"], n, p=[.22,.38,.28,.12])
    tenure0, base_arpu = rng.integers(1,72,n), rng.lognormal(5.6,.55,n)
    heavy_data, eps = rng.random(n) < .30, rng.normal(0,.55,n)

    r1 = (-1.0 - 0.032*np.clip(tenure0,0,48) - 0.0014*base_arpu
          + 0.60*(plan=="Prepaid Basic") + 0.50*(handset=="Feature")
          + 0.30*(region=="Western") + eps)
    r2 = (-0.55 - 0.006*np.clip(tenure0,0,48) + 0.0003*base_arpu + 1.15*heavy_data
          + 0.70*np.isin(handset,["Smart-Mid","Smart-High"]) + 0.20*(plan=="Prepaid Basic") + eps)
    p1, p2 = 1/(1+np.exp(-r1)), 1/(1+np.exp(-r2))

    churn_month = np.full(n, 99)
    for m in range(1, n_months+1):
        hz = np.clip((p1 if m <= 5 else p2)*0.20, 0, .9)
        churn_month[(rng.random(n) < hz) & (churn_month == 99)] = m

    rows = []
    for m in range(1, n_months+1):
        alive = churn_month >= m
        decay = np.where(np.clip(churn_month-m,0,6) <= 2, 0.66 + 0.17*np.clip(churn_month-m,0,6), 1.0)
        rows.append(pd.DataFrame({
            "subscriber_id": np.arange(n), "month": m, "region": region, "plan": plan,
            "handset_class": handset, "tenure_months": tenure0 + m,
            "topup_kes": np.where(alive, base_arpu*decay*rng.lognormal(0,.30,n), 0).round(0),
            "topup_count": np.where(alive, rng.poisson(np.clip(6*decay,0.3,None)), 0),
            "data_mb": np.where(alive, base_arpu*decay*rng.lognormal(0,.5,n)*(1+1.4*heavy_data), 0).round(1),
            "voice_minutes": np.where(alive, rng.gamma(3, 22*decay), 0).round(1),
            "days_since_last_topup": np.where(alive,
                np.clip(rng.poisson(np.clip(20*(1-decay)+4,1,None)),0,60), 60),
            "complaints_90d": rng.poisson(0.18 + 0.45*((churn_month-m) <= 2), n),
            "active": alive.astype(int), "_churn_month": churn_month}))
    return pd.concat(rows, ignore_index=True)

def build_snapshot(panel, prediction_month, horizon=2):
    """Features as at the prediction point; label from the outcome window."""
    obs = panel[(panel.month == prediction_month) & (panel.active == 1)].copy()
    cm = obs._churn_month.values
    obs["churned"] = ((cm > prediction_month) & (cm <= prediction_month + horizon)).astype(int)
    return obs.drop(columns=["_churn_month", "active", "month"])

FEATURES_NUM = ["tenure_months","topup_kes","topup_count","data_mb",
                "voice_minutes","days_since_last_topup","complaints_90d"]
FEATURES_CAT = ["region","plan","handset_class"]

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)

def add_realistic_dirt(snap, seed=0):
    """Inject the problems every real extract arrives with."""
    rng = np.random.default_rng(seed)
    d = snap.copy()
    # 1. Inconsistent categorical labels from three different source systems
    m = rng.random(len(d)) < .18
    d.loc[m & (d.region == "Nairobi"), "region"] = "nairobi"
    m2 = rng.random(len(d)) < .09
    d.loc[m2 & (d.region == "Nairobi"), "region"] = "NBO"
    d.loc[rng.random(len(d)) < .05, "region"] = np.nan
    # 2. Missing values that are NOT missing at random: the billing feed drops
    #    records for subscribers with no activity in the period
    # the billing feed drops the amount when a subscriber has gone
    # more than a week without topping up
    lapsed = d.days_since_last_topup >= 9
    d.loc[lapsed & (rng.random(len(d)) < .75), "topup_kes"] = np.nan
    d.loc[rng.random(len(d)) < .08, "voice_minutes"] = np.nan
    # 3. A unit error: a handful of records in bytes rather than megabytes
    bad = rng.random(len(d)) < .004
    d.loc[bad, "data_mb"] = d.loc[bad, "data_mb"] * 1_048_576
    return d

panel = make_panel()
train = add_realistic_dirt(pd.concat([build_snapshot(panel,3), build_snapshot(panel,4)],
                                     ignore_index=True), seed=1)
test  = add_realistic_dirt(build_snapshot(panel, 6), seed=2)
print(f"train {train.shape}   test {test.shape}")

## 1. Structural fixes: the same thing, spelled three ways

In [ ]:
print(train.region.value_counts(dropna=False).to_string())

In [ ]:
# A mapping is a decision, so write it down explicitly. Never a chain of .replace()
# calls scattered through the notebook.
REGION_MAP = {"nairobi": "Nairobi", "NBO": "Nairobi"}

def fix_region(d):
    d = d.copy()
    d["region"] = d["region"].replace(REGION_MAP)
    return d

train, test = fix_region(train), fix_region(test)
print(train.region.value_counts(dropna=False).to_string())

**Note what just happened.** The fix is a named function applied identically to
train and test. That is not tidiness. It is the only way the transformation you
tested is the transformation that runs in production.

## 2. Missing values: the indicator comes first

The instinct is to fill the gap and move on. That throws away information, because
**the fact that a value is missing is itself a signal**, and here it is a strong one:
the billing feed drops the top-up amount for subscribers who have gone more than a
week without topping up, and those subscribers are disproportionately on their way out.

In [ ]:
# READ: is missingness related to the outcome?
for col in ["topup_kes", "voice_minutes", "region"]:
    miss = train[col].isnull()
    print(f"{col:<16} missing in {miss.mean():>5.1%} of rows | "
          f"churn when present {train.loc[~miss,'churned'].mean():>5.1%} | "
          f"churn when missing {train.loc[miss,'churned'].mean():>5.1%}")

`topup_kes` missingness carries real signal. `voice_minutes` missingness does not,
because we injected it at random. A real extract contains both kinds and you cannot
tell them apart without this check.

**So: create the indicator, then fill.** In that order, always.

In [ ]:
FILL_VALUES = {}      # learned from TRAIN only

def handle_missing(d, fit=False):
    d = d.copy()
    for col in ["topup_kes", "voice_minutes"]:
        d[f"{col}_missing"] = d[col].isnull().astype(int)     # indicator FIRST
        if fit:
            FILL_VALUES[col] = d.loc[d[col].notnull(), col].median()
        d[col] = d[col].fillna(FILL_VALUES[col])              # then fill
    d["region"] = d["region"].fillna("Unknown")
    return d

train = handle_missing(train, fit=True)    # learn the medians here
test  = handle_missing(test,  fit=False)   # apply them unchanged here

print("Medians learned from training data:", {k: round(v) for k, v in FILL_VALUES.items()})
print("Remaining nulls -> train:", train.isnull().sum().sum(), "| test:", test.isnull().sum().sum())

**This is the cell that matters in this notebook.**

`fit=True` on train computes the medians. `fit=False` on test applies them. If you
had computed the median across both, the test set would have influenced a number
used to build the model, and your test score would no longer be an honest estimate
of anything.

Note also the pandas 3.0-safe assignment: `d[col] = d[col].fillna(...)`. The older
`d.col.fillna(x, inplace=True)` is chained assignment on a column, deprecated in
pandas 2.2 and broken under copy-on-write.

## 3. Outliers: a decision, not a deletion

In [ ]:
print(train.data_mb.sort_values(ascending=False).head(4).to_string())
print(f"\nMedian data_mb: {train.data_mb.median():,.0f}")

Those few records are a unit error, not genuine heavy users. There are three
responses, and choosing between them is a judgement you must be able to defend:

| Response | When it is right | What it costs |
|---|---|---|
| **Drop the rows** | Never, if the same records will appear at scoring time | You cannot drop a row in production |
| **Correct the error** | When you know the cause, as here | Requires the cause to be known |
| **Cap at a learned quantile** | When the cause is unknown | The cap is a number learned from data |

Our cause is known, so we correct. **The order matters more than the choice.** A
threshold learned from dirty data inherits the dirt: the 99.9th percentile of
`data_mb` right now is in the hundreds of millions, so a cap derived from it would
never fire and the errors would survive untouched.

Domain rule first. Learned statistic second. Always that way round.

In [ ]:
CAP = {}
PLAUSIBLE_MAX_MB = 500_000     # 500 GB on a prepaid line in one month is impossible

def fix_units_and_cap(d, fit=False):
    d = d.copy()
    # STEP 1 - domain rule first. A statistic learned from dirty data is dirty:
    # the 99.9th percentile of this column is currently in the hundreds of
    # millions, so any cap derived from it would never fire.
    unit_error = d.data_mb > PLAUSIBLE_MAX_MB
    d.loc[unit_error, "data_mb"] = d.loc[unit_error, "data_mb"] / 1_048_576
    # STEP 2 - only now is a learned threshold meaningful
    if fit:
        CAP["data_mb"] = d.data_mb.quantile(0.999)       # learned on TRAIN only
    d["data_mb"] = d["data_mb"].clip(upper=CAP["data_mb"])
    return d

print(f"99.9th percentile BEFORE correcting units: {train.data_mb.quantile(0.999):,.0f} MB")

train = fix_units_and_cap(train, fit=True)
test  = fix_units_and_cap(test,  fit=False)

print(f"Cap learned from CORRECTED training data : {CAP['data_mb']:,.0f} MB")
print(f"Max after correction -> train {train.data_mb.max():,.0f} | test {test.data_mb.max():,.0f}")

## 4. Enrichment

Now we add features the business would recognise. Two rules:

1. Every feature must be computable **at the prediction point**.
2. Any threshold inside a feature is a number learned from data, so it is learned
   from training data only.

In [ ]:
THRESH = {}

def enrich(d, fit=False):
    d = d.copy()
    if fit:
        THRESH["heavy_data"]  = d.data_mb.quantile(0.75)
        THRESH["low_topup"]   = d.topup_kes.quantile(0.25)

    d["heavy_data_user"]  = (d.data_mb   > THRESH["heavy_data"]).astype(int)
    d["low_value"]        = (d.topup_kes < THRESH["low_topup"]).astype(int)
    d["has_complained"]   = (d.complaints_90d >= 1).astype(int)
    d["lapsed"]           = (d.days_since_last_topup > 21).astype(int)
    d["revenue_per_topup"] = d.topup_kes / d.topup_count.clip(lower=1)
    d["smartphone"]       = d.handset_class.isin(["Smart-Mid","Smart-High"]).astype(int)
    return d

train = enrich(train, fit=True)
test  = enrich(test,  fit=False)
print("Thresholds learned from training data:", {k: round(v) for k, v in THRESH.items()})
train[["heavy_data_user","low_value","has_complained","lapsed","smartphone"]].mean().round(3)

### FILL: measure the cost of getting this wrong

In [ ]:
# FILL: this is the same enrichment, but with the threshold learned from
# train and test POOLED - the shortcut almost everyone takes.

pooled = pd.concat([train, test], ignore_index=True)
leaky_threshold  = pooled.data_mb.quantile(0.75)
honest_threshold = THRESH["heavy_data"]

print(f"Threshold from training data only : {honest_threshold:,.0f} MB")
print(f"Threshold from train + test pooled: {leaky_threshold:,.0f} MB")
print(f"Difference: {abs(leaky_threshold - honest_threshold):,.0f} MB "
      f"({abs(leaky_threshold/honest_threshold - 1):.1%})")

flips = (test.data_mb > honest_threshold) != (test.data_mb > leaky_threshold)
print(f"\nTest rows whose heavy_data_user flag flips: {flips.sum():,} of {len(test):,}")

**READ and answer.** The difference looks small, and on this dataset it is.

> *So why is it still wrong?*

Two reasons. The size of the effect is not knowable in advance, so "it was only a
few percent" is a conclusion you can only reach after you have already compromised
the test set. And the habit does not stay small: the same shortcut applied to a
target-encoded categorical, or to a scaler fitted on everything, produces test
scores that are badly and invisibly inflated.

Correctness here is cheap. Discovering it late is not.

## 5. Save the analytical base table

In [ ]:
train.to_csv("abt_train.csv", index=False)
test.to_csv("abt_test.csv",  index=False)

print(f"abt_train.csv  {train.shape}")
print(f"abt_test.csv   {test.shape}")
print(f"\nChurn rate -> train {train.churned.mean():.1%} | test {test.churned.mean():.1%}")
print("\nThese two files are the input to notebook 04.")

### What you should take away

The transformations in this notebook are not complicated. What makes them
professional is that each one is a **named function with a `fit` flag**, learned on
train and applied to test.

That pattern is what `sklearn`'s `Pipeline` automates, and notebook 04 uses the
automated version. Doing it by hand once is how you understand what the pipeline is
protecting you from.

**Next:** `03_Bias_Variance_Simulation`, then `04_Model_Training_and_Selection`.